In [10]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

data = [[3, 'Brad', None, 4000], [1, 'John', 3, 1000], [2, 'Dan', 3, 2000], [4, 'Thomas', 3, 4000]]
employee = pd.DataFrame(data, columns=['empId', 'name', 'supervisor', 'salary']).astype({'empId':'Int64', 'name':'object', 'supervisor':'Int64', 'salary':'Int64'})
data = [[2, 500], [4, 2000]]
bonus = pd.DataFrame(data, columns=['empId', 'bonus']).astype({'empId':'Int64', 'bonus':'Int64'})

# Initialize Spark session
spark = SparkSession.builder \
    .appName("employee") \
    .getOrCreate()

# Create Spark DataFrames
employee_df = spark.createDataFrame(employee)
bonus_df = spark.createDataFrame(bonus)

employee_df.show()
bonus_df.show()

no_bonus_df = (
    employee_df.join(bonus_df, on="empId", how="left_anti")
    .select(col("name"), lit(None).alias("bonus"))
)
print("Employees without bonus:")
no_bonus_df.show()

bonus_only_df = employee_df.alias("e").join(bonus_df.alias("b"), on='empId', how='inner')\
    .filter(col("b.bonus") < 1000)\
    .select("e.name", "b.bonus")
print("Employees with bonus:")
bonus_only_df.show()    

final_df = no_bonus_df.union(bonus_only_df).orderBy("name")
print("Final DataFrame:")
final_df.show()

# Stop Spark session
spark.stop()


+-----+------+----------+------+
|empId|  name|supervisor|salary|
+-----+------+----------+------+
|    3|  Brad|       NaN|  4000|
|    1|  John|       3.0|  1000|
|    2|   Dan|       3.0|  2000|
|    4|Thomas|       3.0|  4000|
+-----+------+----------+------+

+-----+-----+
|empId|bonus|
+-----+-----+
|    2|  500|
|    4| 2000|
+-----+-----+

Employees without bonus:
+----+-----+
|name|bonus|
+----+-----+
|Brad| null|
|John| null|
+----+-----+

Employees with bonus:
+----+-----+
|name|bonus|
+----+-----+
| Dan|  500|
+----+-----+

Final DataFrame:
+----+-----+
|name|bonus|
+----+-----+
|Brad| null|
| Dan|  500|
|John| null|
+----+-----+



In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lit

spark = SparkSession.builder.appName("employee").getOrCreate()

employee_data = [
    (3, 'Brad', None, 4000),
    (1, 'John', 3, 1000),
    (2, 'Dan', 3, 2000),
    (4, 'Thomas', 3, 4000)
]
bonus_data = [
    (2, 500),
    (4, 2000)
]

employee_df = spark.createDataFrame(employee_data, ["empId", "name", "supervisor", "salary"])
bonus_df = spark.createDataFrame(bonus_data, ["empId", "bonus"])


final_df = (
    employee_df.alias("e")
    .join(bonus_df.alias("b"), on="empId", how="left")
    .select(
        col("e.name"),
        when(col("b.empId").isNull(), lit(None))       # no bonus  
        .when(col("b.bonus") < 1000, col("b.bonus"))   # bonus < 1000 
        .alias("bonus")
    )
    .filter(col("b.empId").isNull() | (col("b.bonus") < 1000))   
    .orderBy("name")
)

print("Final DataFrame:")
final_df.show()

spark.stop()

Final DataFrame:
+----+-----+
|name|bonus|
+----+-----+
|Brad| null|
| Dan|  500|
|John| null|
+----+-----+



In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("employee").getOrCreate()

employee_data = [
    (3, 'Brad', None, 4000),
    (1, 'John', 3, 1000),
    (2, 'Dan', 3, 2000),
    (4, 'Thomas', 3, 4000)
]
bonus_data = [
    (2, 500),
    (4, 2000)
]

employee_df = spark.createDataFrame(employee_data, ["empId", "name", "supervisor", "salary"])
bonus_df = spark.createDataFrame(bonus_data, ["empId", "bonus"])
'''
# Left join with filtered bonus
final_df = (
    employee_df.alias("e")
    .join(bonus_df.alias("b"), on="empId", how="left")
    .select(
        col("e.name"),
        when(col("b.empId").isNull(), lit(None))       # no bonus → null
        .when(col("b.bonus") < 1000, col("b.bonus"))   # bonus < 1000 → actual bonus
        .alias("bonus")
    )
    .filter(col("b.empId").isNull() | (col("b.bonus") < 1000))  # exclude ≥1000
    .orderBy("name")
)
'''

# Left join with filtered bonus
final_df = (
    employee_df.alias("e")
    .join(bonus_df.alias("b"), on="empId", how="left")
    .select("e.name", "b.bonus")
    .filter(col("b.empId").isNull() | (col("b.bonus") < 1000))  # exclude ≥1000
    .orderBy("name")
)

print("Final DataFrame:")
final_df.show()

spark.stop()

Final DataFrame:
+----+-----+
|name|bonus|
+----+-----+
|Brad| null|
| Dan|  500|
|John| null|
+----+-----+

